# SPII-Guard demo

Detect sensitive information (dates, times, last 4 card digits, people's names) in a receipt image using Tesseract OCR, regular expressions and spaCy named entity recognition.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Lightningstudy/SPII-Guard/blob/main/SPII_Guard_Demo.ipynb)

Run the cells top to bottom (**Runtime > Run all**). Setup takes about a minute. No GPU needed.

## 1. Install Tesseract and the Python dependencies

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!git clone -q https://github.com/Lightningstudy/SPII-Guard.git
%cd SPII-Guard
!pip install -q -r requirements.txt
!tesseract --version | head -1

## 2. Run the detector on the bundled sample receipt

Red boxes mark the sensitive words. With `--save-ocr` a second image shows every word Tesseract read (blue boxes), which is useful for seeing what the regex and NER stages had to work with.

In [ ]:
!python spii_guard_demo.py --save-ocr

## 3. Look at the result

In [ ]:
from IPython.display import display
from PIL import Image

def side_by_side(paths, height=600):
    imgs = [Image.open(p).convert("RGB") for p in paths]
    imgs = [im.resize((int(im.width * height / im.height), height)) for im in imgs]
    canvas = Image.new("RGB", (sum(im.width for im in imgs) + 20 * (len(imgs) - 1), height), "white")
    x = 0
    for im in imgs:
        canvas.paste(im, (x, 0)); x += im.width + 20
    return canvas

print("Original | every OCR word (blue) | sensitive words (red)")
display(side_by_side([
    "images/originalReceipt.png",
    "output/originalReceipt_ocr.jpg",
    "output/originalReceipt_detect.jpg",
]))

## 4. Try your own receipt

Run the next cell and pick an image (JPG or PNG) from your computer. Results depend heavily on OCR quality, so a sharp, straight, well-lit photo works best.

In [ ]:
from google.colab import files
import os, shlex

uploaded = files.upload()
for name in uploaded:
    !python spii_guard_demo.py --image {shlex.quote(name)} --save-ocr
    stem = os.path.splitext(name)[0]
    display(side_by_side([name, f"output/{stem}_ocr.jpg", f"output/{stem}_detect.jpg"]))

## How it works

1. **OCR**: `pytesseract.image_to_data` (`--oem 3 --psm 6`) returns every word with its bounding box.
2. **Regular expressions** flag masked card numbers (`XXXX...5562`, `xx 1234`, `KK1234`), dates containing `/`, and times containing `:`.
3. **Named entity recognition**: spaCy `en_core_web_sm` tags `PERSON` entities; a list of store names and job titles (Walmart, Cashier, ...) filters out false positives.
4. Boxes for every flagged word are drawn on a copy of the original image.

See `spii_guard_demo.py` for the headless implementation and `src/SPII_Guard.py` for the original research script.